In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#  1. Load & combine match metadata files

match_files = [
    'matches_updated_ipl_upto_2025.csv',
    'matches_updated_mens_ipl.csv'
]

dfs = []
for path in match_files:
    tmp = pd.read_csv(path, low_memory=False)
    tmp.columns = tmp.columns.str.lower().str.strip()
    dfs.append(tmp)

matches = pd.concat(dfs, ignore_index=True, sort=False)

# Remove exact duplicate rows
matches = matches.drop_duplicates()

# Keep last record per matchid (most recent version wins)
matches = matches.sort_values(['matchid', 'date'], ascending=[True, False])
matches = matches.drop_duplicates(subset=['matchid'], keep='first')

# Standardize matchid type
matches['matchid'] = matches['matchid'].astype(str).str.strip()

# Parse date
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')

# Fill / clean critical categorical columns
for col in ['venue', 'season', 'city', 'team1', 'team2', 'winner', 'toss_winner', 'player_of_match']:
    if col in matches.columns:
        matches[col] = matches[col].fillna('Unknown').astype(str)

# Numeric win margin columns
for col in ['winner_runs', 'winner_wickets']:
    if col in matches.columns:
        matches[col] = pd.to_numeric(matches[col], errors='coerce').fillna(0).astype(int)

# Save cleaned match data
matches.to_csv('clean_matches.csv', index=False)
print("clean_matches.csv saved")
print("rows:", len(matches), "  columns:", list(matches.columns))
print(matches[['matchid','date','team1','team2','venue','winner']].tail(3))

clean_matches.csv saved
rows: 1169   columns: ['season', 'venue', 'event', 'winner_runs', 'umpire2', 'toss_winner', 'date', 'neutralvenue', 'umpire1', 'city', 'reserve_umpire', 'winner', 'eliminator', 'date1', 'method', 'team1', 'toss_decision', 'gender', 'team2', 'balls_per_over', 'winner_wickets', 'tv_umpire', 'player_of_match', 'match_referee', 'outcome', 'date2', 'match_number', 'matchid']
      matchid       date                        team1           team2  \
1167  1473510 2025-06-01               Mumbai Indians    Punjab Kings   
1168  1473511 2025-06-03  Royal Challengers Bengaluru    Punjab Kings   
1160  1485779 2025-05-24                 Punjab Kings  Delhi Capitals   

                                 venue                       winner  
1167  Narendra Modi Stadium, Ahmedabad                 Punjab Kings  
1168  Narendra Modi Stadium, Ahmedabad  Royal Challengers Bengaluru  
1160    Sawai Mansingh Stadium, Jaipur               Delhi Capitals  


In [7]:
#  2. Load & combine ball-by-ball files

ball_files = [
    'deliveries_updated_mens_ipl.csv',
    'deliveries_updated_ipl_upto_2025.csv',
    'IPL_ball_by_ball_updated.csv'
]

dfs = []
for path in ball_files:
    tmp = pd.read_csv(path, low_memory=False)
    tmp.columns = tmp.columns.str.lower().str.strip()
    dfs.append(tmp)

deliveries = pd.concat(dfs, ignore_index=True, sort=False)

# Deduplicate rows (same ball should appear only once)
deliveries = deliveries.drop_duplicates(
    subset=['matchid', 'inning', 'over', 'ball'],
    keep='last'
)

# Standardize matchid
deliveries['matchid'] = deliveries['matchid'].astype(str).str.strip()

# Parse date if present
if 'date' in deliveries.columns:
    deliveries['date'] = pd.to_datetime(deliveries['date'], errors='coerce')

# Fill missing team / player names
for col in ['batting_team', 'bowling_team', 'batsman', 'non_striker', 'bowler']:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].fillna('Unknown').astype(str)

# Convert numeric columns safely
numeric_cols = [
    'batsman_runs', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty',
    'iswide', 'isnoball', 'total_runs'
]

for col in numeric_cols:
    if col in deliveries.columns:
        deliveries[col] = pd.to_numeric(deliveries[col], errors='coerce').fillna(0)

# Create is_valid_ball flag (exclude wides & no-balls for balls-faced count)
deliveries['is_valid_ball'] = (
    (deliveries.get('iswide', 0) == 0) &
    (deliveries.get('isnoball', 0) == 0)
).astype(int)

# Compute total_runs if not already present
if 'total_runs' not in deliveries.columns:
    deliveries['total_runs'] = deliveries['batsman_runs'] + deliveries['extras']

# Save cleaned deliveries
deliveries.to_csv('clean_deliveries.csv', index=False)
print("clean_deliveries.csv saved")
print("rows:", len(deliveries), "  columns:", list(deliveries.columns))
print(deliveries[['matchid','inning','over','ball','batsman','bowler','total_runs']].tail(3))

clean_deliveries.csv saved
rows: 197727   columns: ['matchid', 'inning', 'over_ball', 'over', 'ball', 'batting_team', 'bowling_team', 'batsman', 'non_striker', 'bowler', 'batsman_runs', 'extras', 'iswide', 'isnoball', 'byes', 'legbyes', 'penalty', 'dismissal_kind', 'player_dismissed', 'date', 'match_id', 'season', 'start_date', 'venue', 'innings', 'striker', 'runs_off_bat', 'wides', 'noballs', 'wicket_type', 'other_wicket_type', 'other_player_dismissed', 'is_valid_ball', 'total_runs']
       matchid  inning  over  ball  batsman  bowler  total_runs
535375     nan     NaN   NaN  17.4  Unknown  P Negi         0.0
535376     nan     NaN   NaN  17.5  Unknown  P Negi         0.0
535377     nan     NaN   NaN  17.6  Unknown  P Negi         0.0
